# HBCC Standard KD, DKD và DKD + Attention trên Kaggle

Notebook này huấn luyện **HBCC-Small** và **HBCC-Medium** bằng ba phương pháp có thể bật/tắt độc lập:

1. `standard`: KD chuẩn với `KL(teacher || student)`.
2. `dkd`: Decoupled Knowledge Distillation với TCKD và NCKD.
3. `dkd_at`: DKD kết hợp Attention Transfer trên feature Stage 2–4.

Teacher là checkpoint ResNet-18 baseline đã huấn luyện bằng CE. Toàn bộ dữ liệu train/validation/test chỉ dùng `ToTensor + Normalize`; notebook và runner sẽ từ chối teacher hoặc recipe có augmentation. HBCC student luôn được khởi tạo lại từ cùng seed, không warm-start từ checkpoint HBCC-CE.

## HBCC-Wide Stage 4 v1 (đã khôi phục)

Mọi phương pháp đều khởi tạo student mới với đúng kiến trúc CE 84,20% `hbcc_wide_stage4_v1`: Small `[48, 80, 160, 256]`, Medium `[64, 96, 192, 288]`; depth `[1, 1, 2, 1]`. Stage 4 dùng proposal `1×1`, hard assignment, một block Context Cluster thuần và không có nhánh DWConv. Các Stage giữ độ phân giải `32 → 16 → 8 → 4`; PointReducer vẫn là convolution 3×3.

`kd_alpha` chỉ thuộc Standard KD. DKD dùng `DKD_SCALE`, trọng số TCKD/NCKD và warmup. `dkd_at` bổ sung Attention Transfer không cần teacher/student có cùng số channel; hoàn toàn không dùng augmentation.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
import torch
import yaml

## 1. Cấu hình đường dẫn và thực nghiệm

Các đường dẫn quan trọng đều nằm trong ô dưới đây. Trên Kaggle:

- `REPO_ROOT`: thư mục chứa repository. Có thể để `None` nếu notebook nằm trong repository hoặc repo ở `/kaggle/working/Lightweight-Context-Cluster`.
- `DATA_ROOTS`: thư mục mà `torchvision.datasets.CIFAR10/CIFAR100` sử dụng làm `root`.
- `TEACHER_CHECKPOINTS`: mỗi dataset/seed phải trỏ đến `best.pth` của ResNet-18 no-augmentation.
- `TEACHER_CONFIGS`: có thể để `None`; config sẽ được lấy trực tiếp từ checkpoint.
- `REFERENCE_RESULTS_ROOTS`: tùy chọn, dùng để đọc kết quả CE cũ vào bảng so sánh; không dùng để huấn luyện KD.
- `/kaggle/input` là read-only, vì vậy `OUTPUT_ROOT` nên đặt trong `/kaggle/working`.

In [ ]:
# ---------- Duong dan Kaggle can chinh ----------
REPO_ROOT = None  # Vi du: Path('/kaggle/input/lightweight-context-cluster')

DATASETS = ['cifar10']  # Co the dung ['cifar10', 'cifar100']
SEEDS = [42]
DATA_ROOTS = {
    'cifar10': Path('/kaggle/working/data'),
    'cifar100': Path('/kaggle/working/data'),
}
OUTPUT_ROOT = Path('/kaggle/working/hbcc_wide_v1_kd_attention_runs')

TEACHER_CHECKPOINTS = {
    ('cifar10', 42): Path('/kaggle/input/CHANGE-ME/cifar10_resnet18/best.pth'),
    # ('cifar100', 42): Path('/kaggle/input/CHANGE-ME/cifar100_resnet18/best.pth'),
}
TEACHER_CONFIGS = {
    ('cifar10', 42): None,  # Hoac Path('/kaggle/input/.../config.yaml')
    # ('cifar100', 42): None,
}
REFERENCE_RESULTS_ROOTS = [
    # Path('/kaggle/input/CHANGE-ME/baseline-runs'),
]


# ---------- Switch chon student va phuong phap ----------
TRAIN_HBCC_SMALL = False
TRAIN_HBCC_MEDIUM = True
RUN_STANDARD_KD = False
RUN_DKD = False
RUN_DKD_AT = True  # Cau hinh muc tieu: DKD + Attention Stage 2-4

STUDENTS = [
    name for name, enabled in {
        'hbcc_small': TRAIN_HBCC_SMALL,
        'hbcc_medium': TRAIN_HBCC_MEDIUM,
    }.items() if enabled
]
METHODS = [
    name for name, enabled in {
        'standard': RUN_STANDARD_KD,
        'dkd': RUN_DKD,
        'dkd_at': RUN_DKD_AT,
    }.items() if enabled
]
EXPECTED_EMBED_DIMS = {
    'hbcc_small': [48, 80, 160, 256],
    'hbcc_medium': [64, 96, 192, 288],
}
EXPECTED_DEPTHS = [1, 1, 2, 1]
EXPECTED_PROPOSALS = [[2, 2], [2, 2], [2, 2], [1, 1]]
EXPECTED_ASSIGNMENT_MODES = ['hard', 'hard', 'hard', 'hard']
EXPECTED_ASSIGNMENT_TEMPERATURES = [1.0, 1.0, 1.0, 1.0]
EXPECTED_STAGE_MODES = ['hybrid', 'hybrid', 'cluster', 'cluster']
EXPECTED_LOCAL_BRANCHES = ['lbpconv', 'dwconv', 'identity', 'identity']
EXPECTED_LOCAL_RATIOS = [0.5, 0.5, 0.0, 0.0]
EXPECTED_CHANNEL_SHUFFLE = [True, True, False, False]
HBCC_ARCHITECTURE = 'hbcc_wide_stage4_v1'

KD_EPOCHS = 300  # Co the chinh sua doc lap voi epoch teacher
EXPECTED_TEACHER_EPOCHS = 300  # Khop voi CE_EPOCHS mac dinh cua notebook CE
LABEL_SMOOTHING = 0.0  # KD da cung cap soft target; CE notebook van co the dung 0.1
KD_TEMPERATURE = 4.0
STANDARD_KD_ALPHA = 0.5
DKD_TCKD_WEIGHT = 1.0
DKD_NCKD_WEIGHT = 4.0
DKD_SCALE = 0.5
DKD_WARMUP_EPOCHS = 20
FEATURE_KD_WEIGHT = 0.25
FEATURE_KD_STAGES = [2, 3, 4]
FEATURE_KD_WARMUP_EPOCHS = 20
TARGET_GAP_TO_TEACHER = 0.5

DOWNLOAD_DATA = True  # False neu CIFAR da co san trong DATA_ROOTS
NUM_WORKERS = 4
DEVICE = 'auto'
PRINT_EVERY = 5
SHOW_PROGRESS = False
FORCE = False
SMOKE = False  # True: FakeData, 1 epoch va 1 batch de kiem tra luong

In [ ]:
def find_repo_root(explicit_root=None):
    candidates = []
    if explicit_root is not None:
        candidates.append(Path(explicit_root))
    candidates.extend([
        Path.cwd(),
        Path.cwd().parent,
        Path('/kaggle/working/Lightweight-Context-Cluster'),
    ])
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if (candidate / 'tools' / 'run_kd_comparison.py').is_file():
            return candidate
    raise FileNotFoundError('Khong tim thay repository chua tools/run_kd_comparison.py')


ROOT = find_repo_root(REPO_ROOT)
RUNNER = ROOT / 'tools' / 'run_kd_comparison.py'
OUTPUT_ROOT = OUTPUT_ROOT.expanduser().resolve()

assert DATASETS and set(DATASETS) <= {'cifar10', 'cifar100'}
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert STUDENTS and set(STUDENTS) <= {'hbcc_small', 'hbcc_medium'}
assert METHODS and set(METHODS) <= {'standard', 'dkd', 'dkd_at'}
assert isinstance(KD_EPOCHS, int) and KD_EPOCHS > 0
assert KD_TEMPERATURE > 0 and 0 < STANDARD_KD_ALPHA <= 1
assert DKD_TCKD_WEIGHT >= 0 and DKD_NCKD_WEIGHT >= 0
assert DKD_TCKD_WEIGHT + DKD_NCKD_WEIGHT > 0
assert DKD_SCALE > 0
assert isinstance(DKD_WARMUP_EPOCHS, int) and DKD_WARMUP_EPOCHS >= 0
assert FEATURE_KD_WEIGHT > 0
assert FEATURE_KD_STAGES and len(FEATURE_KD_STAGES) == len(set(FEATURE_KD_STAGES))
assert all(stage in {1, 2, 3, 4} for stage in FEATURE_KD_STAGES)
assert isinstance(FEATURE_KD_WARMUP_EPOCHS, int) and FEATURE_KD_WARMUP_EPOCHS >= 0
assert TARGET_GAP_TO_TEACHER >= 0
assert 0 <= LABEL_SMOOTHING < 1

required_keys = {(dataset, seed) for dataset in DATASETS for seed in SEEDS}
missing_checkpoints = sorted(required_keys - set(TEACHER_CHECKPOINTS))
if missing_checkpoints:
    raise KeyError(f'Thieu TEACHER_CHECKPOINTS cho: {missing_checkpoints}')
for key in sorted(required_keys):
    checkpoint = Path(TEACHER_CHECKPOINTS[key]).expanduser()
    if not checkpoint.is_file():
        raise FileNotFoundError(f'Khong tim thay teacher checkpoint cho {key}: {checkpoint}')
    teacher_config = TEACHER_CONFIGS.get(key)
    if teacher_config is not None and not Path(teacher_config).expanduser().is_file():
        raise FileNotFoundError(f'Khong tim thay teacher config cho {key}: {teacher_config}')

print('Repository  :', ROOT)
print('Python      :', sys.executable)
print('PyTorch     :', torch.__version__)
print('CUDA        :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU         :', torch.cuda.get_device_name(0))
print('Output root :', OUTPUT_ROOT)
print('Datasets    :', DATASETS)
print('Seeds       :', SEEDS)
print('Students    :', STUDENTS)
print('Methods     :', METHODS)
print('Architecture:', HBCC_ARCHITECTURE)
print('Embed dims  :', {name: EXPECTED_EMBED_DIMS[name] for name in STUDENTS})
print('Depths      :', EXPECTED_DEPTHS)
print('DKD scale   :', DKD_SCALE)
print('AT stages   :', FEATURE_KD_STAGES)
print('Target gap  :', TARGET_GAP_TO_TEACHER)
print('KD epochs   :', KD_EPOCHS)
print('Smoke       :', SMOKE)

## 2. Kiểm tra checkpoint và giao thức no-augmentation

Preflight tải checkpoint trên CPU, kiểm tra state dict, config nhúng, ResNet-18, dataset, seed, epoch và `augmentation: none`. Runner dựng forward thử teacher/student, đồng thời xác nhận bốn feature map đều có resolution `32/16/8/4`. Nếu catalog còn kiến trúc HBCC cũ hoặc feature không khớp, notebook dừng trước khi train.

In [ ]:
def build_command(dataset, seed, validate_only=False):
    key = (dataset, seed)
    command = [
        sys.executable, str(RUNNER),
        '--dataset', dataset,
        '--seed', str(seed),
        '--students', *STUDENTS,
        '--methods', *METHODS,
        '--teacher-checkpoint', str(Path(TEACHER_CHECKPOINTS[key]).expanduser().resolve()),
        '--data-root', str(Path(DATA_ROOTS[dataset]).expanduser().resolve()),
        '--output', str(OUTPUT_ROOT),
        '--python', sys.executable,
        '--device', DEVICE,
        '--epochs', str(KD_EPOCHS),
        '--temperature', str(KD_TEMPERATURE),
        '--standard-alpha', str(STANDARD_KD_ALPHA),
        '--dkd-tckd-weight', str(DKD_TCKD_WEIGHT),
        '--dkd-nckd-weight', str(DKD_NCKD_WEIGHT),
        '--dkd-scale', str(DKD_SCALE),
        '--dkd-warmup-epochs', str(DKD_WARMUP_EPOCHS),
        '--feature-kd-weight', str(FEATURE_KD_WEIGHT),
        '--feature-kd-stages', *[str(stage) for stage in FEATURE_KD_STAGES],
        '--feature-kd-warmup-epochs', str(FEATURE_KD_WARMUP_EPOCHS),
        '--label-smoothing', str(LABEL_SMOOTHING),
        '--workers', str(NUM_WORKERS),
        '--print-every', str(PRINT_EVERY),
        '--download-data' if DOWNLOAD_DATA else '--no-download-data',
    ]
    teacher_config = TEACHER_CONFIGS.get(key)
    if teacher_config is not None:
        command.extend(['--teacher-config', str(Path(teacher_config).expanduser().resolve())])
    if EXPECTED_TEACHER_EPOCHS is not None:
        command.extend(['--expected-teacher-epochs', str(EXPECTED_TEACHER_EPOCHS)])
    if SHOW_PROGRESS:
        command.append('--progress')
    if SMOKE:
        command.append('--smoke')
    if FORCE:
        command.append('--force')
    if validate_only:
        command.append('--validate-only')
    return command


def run_command(command):
    print('\n>', subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


for dataset in DATASETS:
    for seed in SEEDS:
        run_command(build_command(dataset, seed, validate_only=True))

## 3. Chạy phương pháp đã bật

Mặc định notebook chỉ chạy HBCC-Medium với `dkd_at` để tập trung vào cấu hình mục tiêu. Có thể bật Standard KD/DKD và HBCC-Small để làm ablation. Run hoàn tất và có config tương thích sẽ được bỏ qua; `FORCE=True` cho phép chạy lại đúng tên run.

In [ ]:
for dataset in DATASETS:
    for seed in SEEDS:
        run_command(build_command(dataset, seed, validate_only=False))

## 4. Tổng hợp kết quả

Bảng chính đọc kết quả Standard KD/DKD vừa chạy. Nếu thêm thư mục vào `REFERENCE_RESULTS_ROOTS`, bảng cũng đọc các run CE no-augmentation trước đó để đối chiếu; các checkpoint CE chỉ được đọc metadata, không được nạp vào student KD.

In [ ]:
records = []
search_roots = [OUTPUT_ROOT, *[Path(path).expanduser().resolve() for path in REFERENCE_RESULTS_ROOTS]]
seen_metrics = set()
for search_root in search_roots:
    if not search_root.exists():
        print('Bo qua reference root khong ton tai:', search_root)
        continue
    for metrics_path in sorted(search_root.rglob('test_metrics.json')):
        metrics_path = metrics_path.resolve()
        if metrics_path in seen_metrics:
            continue
        seen_metrics.add(metrics_path)
        config_path = metrics_path.parent / 'config.yaml'
        if not config_path.is_file():
            continue
        cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
        metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        protocol = cfg.get('protocol', {})
        train_cfg = cfg.get('train', {})
        experiment = cfg.get('experiment', {})
        distillation = cfg.get('distillation', {})
        dataset = protocol.get('dataset')
        seed = int(train_cfg.get('seed', -1))
        model = experiment.get('model_key', cfg.get('model', {}).get('name'))
        augmentation = protocol.get('augmentation')
        method = distillation.get('method', train_cfg.get('kd_method', 'none'))
        method = 'ce' if method in {None, 'none', 'ce'} else method
        expected_epochs = 1 if SMOKE else KD_EPOCHS
        run_epochs = int(train_cfg.get('epochs', -1))
        if dataset not in DATASETS or seed not in SEEDS or augmentation != 'none':
            continue
        if run_epochs != expected_epochs:
            continue
        if method not in {'ce', 'standard', 'dkd', 'dkd_at'}:
            continue
        if method == 'ce' and model not in {*STUDENTS, 'resnet18'}:
            continue
        if method == 'ce' and model in STUDENTS:
            # Run CE 84.20% cũ có thể chưa có architecture tag; toàn bộ trường
            # model bên dưới vẫn phải khớp chính xác HBCC-Wide v1.
            if experiment.get('architecture') not in {None, HBCC_ARCHITECTURE}:
                continue
            if cfg.get('model', {}).get('embed_dims') != EXPECTED_EMBED_DIMS[model]:
                continue
            if cfg.get('model', {}).get('depths') != EXPECTED_DEPTHS:
                continue
            if cfg.get('model', {}).get('proposals') != EXPECTED_PROPOSALS:
                continue
            if cfg.get('model', {}).get('assignment_modes') != EXPECTED_ASSIGNMENT_MODES:
                continue
            if cfg.get('model', {}).get('assignment_temperatures') != EXPECTED_ASSIGNMENT_TEMPERATURES:
                continue
            if cfg.get('model', {}).get('stage_modes') != EXPECTED_STAGE_MODES:
                continue
            if cfg.get('model', {}).get('local_branches') != EXPECTED_LOCAL_BRANCHES:
                continue
            if cfg.get('model', {}).get('local_ratios') != EXPECTED_LOCAL_RATIOS:
                continue
            if cfg.get('model', {}).get('channel_shuffle') != EXPECTED_CHANNEL_SHUFFLE:
                continue
        if method in {'standard', 'dkd', 'dkd_at'}:
            if model not in STUDENTS:
                continue
            if experiment.get('architecture') != HBCC_ARCHITECTURE:
                continue
            if cfg.get('model', {}).get('embed_dims') != EXPECTED_EMBED_DIMS[model]:
                continue
            if cfg.get('model', {}).get('depths') != EXPECTED_DEPTHS:
                continue
            if cfg.get('model', {}).get('proposals') != EXPECTED_PROPOSALS:
                continue
            if cfg.get('model', {}).get('stage_modes') != EXPECTED_STAGE_MODES:
                continue
            if cfg.get('model', {}).get('assignment_modes') != EXPECTED_ASSIGNMENT_MODES:
                continue
            if cfg.get('model', {}).get('assignment_temperatures') != EXPECTED_ASSIGNMENT_TEMPERATURES:
                continue
            if cfg.get('model', {}).get('local_branches') != EXPECTED_LOCAL_BRANCHES:
                continue
            if cfg.get('model', {}).get('local_ratios') != EXPECTED_LOCAL_RATIOS:
                continue
            if cfg.get('model', {}).get('channel_shuffle') != EXPECTED_CHANNEL_SHUFFLE:
                continue
            expected_teacher = Path(TEACHER_CHECKPOINTS[(dataset, seed)]).expanduser().resolve()
            actual_teacher = Path(distillation.get('teacher_checkpoint', '')).expanduser().resolve()
            if actual_teacher != expected_teacher:
                continue
            if float(distillation.get('temperature', -1)) != float(KD_TEMPERATURE):
                continue
            if float(train_cfg.get('label_smoothing', -1)) != float(LABEL_SMOOTHING):
                continue
        if method == 'standard' and float(distillation.get('alpha', -1)) != float(STANDARD_KD_ALPHA):
            continue
        if method in {'dkd', 'dkd_at'}:
            dkd_matches = (
                float(distillation.get('dkd_tckd_weight', -1)) == float(DKD_TCKD_WEIGHT)
                and float(distillation.get('dkd_nckd_weight', -1)) == float(DKD_NCKD_WEIGHT)
                and float(distillation.get('dkd_scale', -1)) == float(DKD_SCALE)
                and int(distillation.get('dkd_warmup_epochs', -1)) == int(DKD_WARMUP_EPOCHS)
            )
            if not dkd_matches:
                continue
        if method == 'dkd_at':
            feature_matches = (
                distillation.get('feature_kd_method') == 'attention'
                and float(distillation.get('feature_kd_weight', -1)) == float(FEATURE_KD_WEIGHT)
                and list(distillation.get('feature_kd_stages', [])) == list(FEATURE_KD_STAGES)
                and int(distillation.get('feature_kd_warmup_epochs', -1)) == int(FEATURE_KD_WARMUP_EPOCHS)
            )
            if not feature_matches:
                continue
        records.append({
            'dataset': dataset,
            'seed': seed,
            'model': model,
            'method': method,
            'epochs': run_epochs,
            'architecture': experiment.get('architecture'),
            'final_features': cfg.get('model', {}).get('embed_dims', [None])[-1],
            'temperature': distillation.get('temperature'),
            'standard_alpha': distillation.get('alpha') if method == 'standard' else None,
            'tckd_weight': distillation.get('dkd_tckd_weight') if method in {'dkd', 'dkd_at'} else None,
            'nckd_weight': distillation.get('dkd_nckd_weight') if method in {'dkd', 'dkd_at'} else None,
            'dkd_scale': distillation.get('dkd_scale') if method in {'dkd', 'dkd_at'} else None,
            'feature_weight': distillation.get('feature_kd_weight') if method == 'dkd_at' else None,
            'feature_stages': distillation.get('feature_kd_stages') if method == 'dkd_at' else None,
            'test_acc1': float(metrics['test_acc1']),
            'test_acc5': metrics.get('test_acc5'),
            'run': metrics_path.parent.name,
            'path': str(metrics_path.parent),
        })

summary = pd.DataFrame(records)
if summary.empty:
    raise RuntimeError('Khong tim thay ket qua test phu hop')
summary = summary.sort_values(['dataset', 'seed', 'model', 'method']).reset_index(drop=True)
teacher_rows = summary[(summary['model'] == 'resnet18') & (summary['method'] == 'ce')]
teacher_lookup = {
    (row.dataset, int(row.seed)): float(row.test_acc1)
    for row in teacher_rows.itertuples()
}
summary['teacher_acc1'] = [
    teacher_lookup.get((row.dataset, int(row.seed)))
    for row in summary.itertuples()
]
summary['gap_to_teacher'] = summary['teacher_acc1'] - summary['test_acc1']
summary['within_target_gap'] = summary['gap_to_teacher'].apply(
    lambda gap: None if pd.isna(gap) else bool(gap <= TARGET_GAP_TO_TEACHER)
)
if not teacher_lookup:
    print('Chua co CE ResNet-18 reference; them thu muc CE vao REFERENCE_RESULTS_ROOTS de tinh gap.')
kd_summary = summary[summary['method'].isin(METHODS)].copy()
expected_kd_runs = len(DATASETS) * len(SEEDS) * len(STUDENTS) * len(METHODS)
if len(kd_summary) != expected_kd_runs:
    raise RuntimeError(f'Ket qua KD chua du: {len(kd_summary)}/{expected_kd_runs} run')
summary_path = OUTPUT_ROOT / 'hbcc_kd_attention_summary.csv'
summary.to_csv(summary_path, index=False)
print('Da luu:', summary_path)
display(summary)
display(summary.pivot_table(index=['dataset', 'seed', 'model'], columns='method', values='test_acc1'))